In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from concurrent.futures import ThreadPoolExecutor


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Đường dẫn đến dữ liệu đã được chia sẵn
data_train_path = "/content/drive/MyDrive/Gym/data/angles_output/train"
data_val_path   = "/content/drive/MyDrive/Gym/data/angles_output/val"
data_test_path  = "/content/drive/MyDrive/Gym/data/angles_output/test"

# Siêu tham số
learning_rate = 1e-4
epochs = 200
patience = 20
mask_value = 0
loss = 'sparse_categorical_crossentropy'

# Các giá trị thử nghiệm cho batch_size và sequence_length
batch_sizes = [8]
sequence_lengths = [30]
overlaps = [0.5]


In [5]:
# Hàm trợ giúp để xử lý từng file CSV, dùng để load dữ liệu nhanh hơn
def process_file(file_path, sequence_length, overlap, label):
    # Bỏ cột đầu tiên nếu không cần thiết (giống như code gốc)
    df = pd.read_csv(file_path).iloc[:, 1:]
    total_rows = df.shape[0]
    segments = []
    # Cắt đoạn dữ liệu với bước nhảy là (sequence_length - overlap)
    for start_idx in range(0, total_rows, sequence_length - overlap):
        end_idx = start_idx + sequence_length
        if end_idx > total_rows:
            break
        segment = df.iloc[start_idx:end_idx].values
        segments.append(segment)
    return segments, [label] * len(segments)

# Hàm load dữ liệu sử dụng đa luồng để tăng tốc độ đọc file
def load_data_with_fixed_length(base_path, sequence_length, overlap, num_workers=4):
    X, y = [], []
    # Lấy danh sách các thư mục con (mỗi thư mục là 1 hành động)
    actions = sorted(os.listdir(base_path))
    tasks = []
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        for label, action in enumerate(actions):
            action_path = os.path.join(base_path, action)
            if os.path.isdir(action_path):
                for file in os.listdir(action_path):
                    file_path = os.path.join(action_path, file)
                    if file.endswith('.csv'):
                        tasks.append(executor.submit(process_file, file_path, sequence_length, overlap, label))
    for future in tasks:
        segments, labels = future.result()
        X.extend(segments)
        y.extend(labels)
    return np.array(X), np.array(y)


In [ ]:
for batch_size in batch_sizes:
    for seq_len in sequence_lengths:
        for overlap in overlaps:
            overlap = int(overlap * seq_len)

            print(f"\nTraining with batch_size={batch_size}, sequence_length={seq_len}, overlap={overlap}")

            # Load dữ liệu từ các folder đã chia sẵn
            X_train, y_train = load_data_with_fixed_length(data_train_path, seq_len, overlap, num_workers=8)
            X_val, y_val     = load_data_with_fixed_length(data_val_path, seq_len, 0, num_workers=8)
            X_test, y_test   = load_data_with_fixed_length(data_test_path, seq_len, 0, num_workers=8)

            # # Tiền xử lý: Scale dữ liệu sử dụng MinMaxScaler
            # scaler = MinMaxScaler()
            # X_train = scaler.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
            # X_val   = scaler.transform(X_val.reshape(-1, 1)).reshape(X_val.shape)
            # X_test  = scaler.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)

            # Xử lý NaN
            X_train = np.nan_to_num(X_train, nan=mask_value)
            X_val   = np.nan_to_num(X_val, nan=mask_value)
            X_test  = np.nan_to_num(X_test, nan=mask_value)

            # # Tính class weight để cân bằng các lớp
            # class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
            # class_weights = dict(enumerate(class_weights))

            # Xây dựng mô hình LSTM
            model = Sequential([
                LSTM(64, return_sequences=True, input_shape=(seq_len, X_train.shape[2])),
                LSTM(64),
                Dense(32, activation='relu'),
                Dropout(0.5),
                Dense(len(np.unique(y_train)), activation='softmax')
            ])

            model.compile(optimizer=Adam(learning_rate=learning_rate),
                          loss=loss,
                          metrics=['accuracy'])

            early_stopping = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

            # Huấn luyện mô hình
            history = model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=batch_size,
                validation_data=(X_val, y_val),
                # class_weight=class_weights,
                callbacks=[early_stopping],
                verbose=1
            )

            # Lưu mô hình
            model_name = f"lstm_model_bs{batch_size}_sl{seq_len}_ol{overlap}.keras"
            save_path = os.path.join("/content/drive/MyDrive/Gym/Model", model_name)
            model.save(save_path)
            print(f"Model saved to {save_path}")

            # Đánh giá trên tập test
            test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
            print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")



Training with batch_size=8, sequence_length=30, overlap=15
